In [13]:
!pip install gradio
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "gradio"
])
import sys
!{sys.executable} -m pip install openai


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
import glob
# from dotenv import load_dotenv
from pathlib import Path
import gradio as gr 
from openai import OpenAI

In [15]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [16]:
from pathlib import Path

knowledge = {}

kb_path = Path(
    r"D:\project\RAG implementation\Code\telecom_demo_kb_large\documents"
)

# Get ONLY files, never folders
filenames = [
    p for p in kb_path.rglob("*")
    if p.is_file()
]

print(f"Found {len(filenames)} files")

# Show what was found
for p in filenames[:10]:
    print(p)

# Load only text files
for filename in filenames:
    

    name = filename.stem

    with filename.open("r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

print(f"Loaded {len(knowledge)} documents")

Found 581 files
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_001.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_002.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_003.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_004.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_005.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_006.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_007.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_008.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_009.md
D:\project\RAG implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_0

In [17]:
knowledge

{'alarm_guide_001': '# Alarm Guide: BGP-NEIGHBOR-FLAP\n\n## Alarm meaning\n\nBGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an observation, not an automatic root-cause classification.\n\n## What commonly correlates\n\nRelated alarms may include LINK-DOWN, HIGH-MEMORY, PACKET-LOSS. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n## False positives and secondary symptoms\n\nThe alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault. Check change windows and neighboring elements before escalating solely from alarm severity.\n\n## Recommended context\n\nAn RCA agent should retrieve the affected device, interface or peer, site, service dependencies, current telemetry, recent changes, and historical incidents matching the alarm and platform.\n\n## Closure evidence\n\nClosure should record why the alarm occurred, what corrected it, whet

In [18]:

knowledge.keys()

dict_keys(['alarm_guide_001', 'alarm_guide_002', 'alarm_guide_003', 'alarm_guide_004', 'alarm_guide_005', 'alarm_guide_006', 'alarm_guide_007', 'alarm_guide_008', 'alarm_guide_009', 'alarm_guide_010', 'alarm_guide_011', 'alarm_guide_012', 'alarm_guide_013', 'alarm_guide_014', 'alarm_guide_015', 'alarm_guide_016', 'alarm_guide_017', 'alarm_guide_018', 'alarm_guide_019', 'alarm_guide_020', 'alarm_guide_021', 'alarm_guide_022', 'alarm_guide_023', 'alarm_guide_024', 'alarm_guide_025', 'alarm_guide_026', 'alarm_guide_027', 'alarm_guide_028', 'alarm_guide_029', 'alarm_guide_030', 'alarm_guide_031', 'alarm_guide_032', 'alarm_guide_033', 'alarm_guide_034', 'alarm_guide_035', 'alarm_guide_036', 'alarm_guide_037', 'alarm_guide_038', 'alarm_guide_039', 'alarm_guide_040', 'alarm_guide_041', 'alarm_guide_042', 'alarm_guide_043', 'alarm_guide_044', 'alarm_guide_045', 'alarm_guide_046', 'alarm_guide_047', 'alarm_guide_048', 'alarm_guide_049', 'alarm_guide_050', 'change_context_001', 'change_context_0

In [19]:
SYSTEM_PREFIX = """You are an AI-assisted Root Cause Analysis (RCA) Agent for a telecom Network Operations Center (NOC).

Your primary responsibility is to analyze network and IT incidents using the provided knowledge base, incident information, 
alarms, telemetry, logs, topology context, historical incidents, postmortems, runbooks, known errors, vendor troubleshooting guides,
 change records, and performance baselines. Politely decline irrelevant question. IF there is not relevant context in the knowladge base. say so instead of making assumptions
 Relevant context:
 """

In [20]:
import re
def get_relevant_context(message):

    text = message.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    words = text.lower().split()
    print(words)
    return [knowledge[word] for word in words if word in knowledge]

In [21]:
get_relevant_context("what is in alarm_guide_001")

['what', 'is', 'in', 'alarm_guide_001']


['# Alarm Guide: BGP-NEIGHBOR-FLAP\n\n## Alarm meaning\n\nBGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an observation, not an automatic root-cause classification.\n\n## What commonly correlates\n\nRelated alarms may include LINK-DOWN, HIGH-MEMORY, PACKET-LOSS. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n## False positives and secondary symptoms\n\nThe alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault. Check change windows and neighboring elements before escalating solely from alarm severity.\n\n## Recommended context\n\nAn RCA agent should retrieve the affected device, interface or peer, site, service dependencies, current telemetry, recent changes, and historical incidents matching the alarm and platform.\n\n## Closure evidence\n\nClosure should record why the alarm occurred, what corrected it, whether dependent servi

In [22]:
def get_additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "No relevant context was found"
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result


In [23]:
print(get_additional_context("what is alarm_guide_001"))

['what', 'is', 'alarm_guide_001']
The following additional context might be relevant in answering the user's question:

# Alarm Guide: BGP-NEIGHBOR-FLAP

## Alarm meaning

BGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an observation, not an automatic root-cause classification.

## What commonly correlates

Related alarms may include LINK-DOWN, HIGH-MEMORY, PACKET-LOSS. Correlation across time and topology can reveal whether the alarm is primary or downstream.

## False positives and secondary symptoms

The alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault. Check change windows and neighboring elements before escalating solely from alarm severity.

## Recommended context

An RCA agent should retrieve the affected device, interface or peer, site, service dependencies, current telemetry, recent changes, and historical incidents matching the alarm and platform.

## Closure 

In [24]:



def chat(message, history):
    system_message = SYSTEM_PREFIX + get_additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history+  [{"role": "user", "content": message}]
    response = client.chat.completions.create(model = MODEL, messages = messages)
    return response.choices[0].message.content




In [25]:
view = gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
